# B-ZTA: Blockchain-Enhanced Zero Trust Architecture (v4.1)

A Python simulation built for my Master of Applied Technologies thesis at Unitec Institute of Technology. It models Zero Trust access control for Bangladesh government digital services, with every access decision written to a signed, tamper-evident blockchain audit log.

**What it demonstrates**
- A Zero Trust gateway that checks identity (MFA), device posture and agency segmentation on every request, following NIST SP 800-207 principles
- A weighted risk engine with behavioural signals (semantic velocity, reputation EMA) and enforced step-up authentication
- Just-in-time (JIT) admin access and trust decay over a session
- An audit chain where each entry is ECDSA-signed and replicated to a second node, so tampering and fabrication are detectable
- Four threat personas: external attacker, malicious insider, privileged admin and stolen credentials

Tags such as **[B1]** or **[C3]** link each feature to a recommendation from the thesis expert interviews. All users, IP addresses and records are synthetic.

Run the cells in order. The last section runs self-checks that verify the main claims.

## Setup and configuration

In [1]:
import copy, hashlib, json, time, random, datetime, math
from dataclasses import dataclass, asdict
from typing import Optional, Dict, List, Tuple
from collections import defaultdict

try:
    from cryptography.hazmat.primitives.asymmetric import ec
    from cryptography.hazmat.primitives import hashes as _ch, serialization
    from cryptography.hazmat.backends import default_backend
    from cryptography.exceptions import InvalidSignature
    ECDSA_OK = True
except ImportError:
    ECDSA_OK = False
    print("WARNING: 'cryptography' is not installed, so ECDSA signing is disabled. "
          "Run: pip install cryptography")

random.seed(42)  # reproducible session and token IDs

USE_COLOR = False  # set True for coloured output in a terminal
if USE_COLOR:
    GREEN, RED, YELLOW = "\033[92m", "\033[91m", "\033[93m"
    CYAN, BOLD, RESET  = "\033[96m", "\033[1m",  "\033[0m"
    DIM, MAGENTA, WHITE = "\033[2m", "\033[95m", "\033[97m"
    ORANGE = "\033[38;5;208m"
else:
    GREEN = RED = YELLOW = CYAN = BOLD = RESET = DIM = MAGENTA = WHITE = ORANGE = ""

DATA_CLASS = {
    "public_announcements":   ("PUBLIC",       1),
    "staff_directory":        ("INTERNAL",     2),
    "health_records":         ("CONFIDENTIAL", 3),
    "tax_records":            ("CONFIDENTIAL", 3),
    "passport_db":            ("CONFIDENTIAL", 3),
    "land_registry":          ("CONFIDENTIAL", 3),
    "financial_transactions": ("SECRET",       4),
    "criminal_records":       ("SECRET",       4),
    "voter_registry":         ("SECRET",       4),
    "citizen_nid_db":         ("TOP_SECRET",   5),
    "intelligence_reports":   ("TOP_SECRET",   5),
    "military_personnel_db":  ("TOP_SECRET",   5),
}
CLASS_COLOR = {
    "PUBLIC": GREEN, "INTERNAL": CYAN, "CONFIDENTIAL": YELLOW,
    "SECRET": ORANGE, "TOP_SECRET": RED,
}
# Illustrative IP-prefix risk map for the simulation only.
# A real deployment would use GeoIP and threat-intelligence feeds.
GEO_RISK = {
    "10.":  ("Bangladesh Internal", 0), "172.": ("Gov VPN",           0),
    "192.": ("Private Network",     1), "103.": ("Bangladesh ISP",     1),
    "202.": ("Asia Pacific",        2), "45.":  ("Suspicious Range",   4),
    "203.": ("Known Threat Actor",  5), "185.": ("Tor Exit Node",      5),
    "194.": ("Eastern Europe",      3),
}

def geo_risk(ip):
    for prefix, info in GEO_RISK.items():
        if ip.startswith(prefix): return info
    return ("Unknown", 3)

## 1. ECDSA key registry [B1]

Each identity gets a SECP256R1 key pair. Log entries are signed by the requesting user, so an administrator with access to a logging node cannot write a convincing fake entry without that user's private key. Keys are held in memory for the simulation; a production system would keep them in an HSM or on smart cards.

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# [B1] ECDSA KEY REGISTRY
# Interview 2 Q3: "Without transaction signing a blockchain audit log is
# tamper-evident against outsiders but not against insiders with node access."
# ─────────────────────────────────────────────────────────────────────────────
class ECDSAKeyRegistry:
    def __init__(self):
        self._priv: Dict = {}
        self._pub:  Dict = {}

    def register(self, uid: str):
        if not ECDSA_OK: return
        key = ec.generate_private_key(ec.SECP256R1(), default_backend())
        self._priv[uid] = key
        self._pub[uid]  = key.public_key()

    def sign(self, uid: str, data: bytes) -> str:
        if not ECDSA_OK or uid not in self._priv: return "NO_ECDSA_LIB"
        return self._priv[uid].sign(data, ec.ECDSA(_ch.SHA256())).hex()

    def verify(self, uid: str, data: bytes, sig_hex: str) -> bool:
        if not ECDSA_OK or uid not in self._pub:
            return sig_hex == "NO_ECDSA_LIB"
        try:
            self._pub[uid].verify(bytes.fromhex(sig_hex), data, ec.ECDSA(_ch.SHA256()))
            return True
        except (InvalidSignature, ValueError):
            return False

    def has_key(self, uid: str) -> bool: return uid in self._pub

    def pubkey_short(self, uid: str) -> str:
        if not ECDSA_OK or uid not in self._pub: return "NOT_REGISTERED"
        raw = self._pub[uid].public_bytes(serialization.Encoding.X962,
              serialization.PublicFormat.CompressedPoint).hex()
        return raw[:16] + "..."

## 2. Trust decay [C4 + B5]

Trust from the initial check decays logarithmically over the session (a fast early drop that slows over time), with an extra penalty per anomaly. With these parameters, a session that starts at 0.92 trust falls below the gateway's 0.70 device-trust threshold after about 85 minutes, which forces re-authentication. The parameters are illustrative, not empirically calibrated.

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# [C4 + B5] TRUST DECAY
# Interview 1 Q2: "A user who passes ZTA at 09:00 and connects to a public
# network at 10:30 receives no re-evaluation under your current design."
# ─────────────────────────────────────────────────────────────────────────────
DECAY_COEFF     = 0.25  # trust lost per unit of log(1 + hours elapsed)
ANOMALY_PENALTY = 0.03  # trust lost per anomaly flag

def decay_trust(base: float, elapsed_min: float, anomalies: int = 0) -> float:
    """Logarithmic trust decay plus a fixed penalty per anomaly flag.
    With base 0.92 the result crosses the 0.70 threshold at ~85 minutes."""
    drop = DECAY_COEFF*math.log1p(elapsed_min/60.0) + ANOMALY_PENALTY*anomalies
    return max(0.0, round(base - drop, 3))

## 3. Core data structures

`ZTALogEntry` is one audit record. `Block` wraps it with the previous block's hash, an ECDSA signature and a small proof-of-work nonce (difficulty 2, illustrative only).

In [4]:
@dataclass
class ZTALogEntry:
    event_type: str; user_id: str; agency: str; resource: str
    ip_address: str; geo_location: str; device_trust_score: float
    data_classification: str; risk_score: float; risk_level: str
    risk_factors: str; decision: str; zt_verdict: str; zt_deny_reason: str
    timestamp: str; session_id: str
    data_volume_kb: float = 0.0; anomaly_flags: str = ""
    policy_version: str = "v4.1-BZTA"; policy_data: str = ""
    jit_token: str = ""; step_up_result: str = ""; threat_persona: str = ""

@dataclass
class ZTAIdentityCheck:
    user_id: str; agency: str; resource: str
    mfa_passed: bool; mfa_method: str; mfa_detail: str
    device_compliant: bool; device_issues: list
    segmentation_allowed: bool; seg_detail: str
    zt_verdict: str; deny_reason: str

class Block:
    def __init__(self, index, log_entry, previous_hash, signature="", signer_id="SYSTEM"):
        self.index = index; self.timestamp = time.time()
        self.log_entry = log_entry; self.previous_hash = previous_hash
        self.signature = signature; self.signer_id = signer_id
        self.nonce = 0; self.hash = self._compute_hash()

    def _compute_hash(self):
        payload = json.dumps({
            "index": self.index, "timestamp": self.timestamp,
            "log_entry": asdict(self.log_entry), "previous_hash": self.previous_hash,
            "nonce": self.nonce, "signature": self.signature, "signer_id": self.signer_id,
        }, sort_keys=True)
        return hashlib.sha256(payload.encode()).hexdigest()

    def mine(self, difficulty=2):
        target = "0"*difficulty
        while not self.hash.startswith(target):
            self.nonce += 1; self.hash = self._compute_hash()

    def tamper(self, field, value): setattr(self.log_entry, field, value)

## 4. Blockchain nodes and consensus [B2, C7]

Two nodes, run by different agencies, each keep their **own copy** of the chain. New blocks follow a simplified PBFT-style flow: the primary proposes a block, the replica checks the hash link and ECDSA signature, and both commit. Because each node stores separate copies, tampering on one node is exposed by comparing it with the other.

**Limitation:** with only two nodes this tolerates no Byzantine (malicious) node faults. Real PBFT needs at least 3f+1 nodes to tolerate f faulty ones, for example four nodes to tolerate one. The simulation demonstrates the message flow and replica-side validation.

A signed heartbeat block is inserted after every 7 regular blocks as a completeness anchor.

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# [B2] DUAL-NODE PBFT-STYLE CONSENSUS
# Interview 2 Q7: "PBFT gives deterministic finality — the wrong choice for
# an audit log needing real-time alerts is PoW probabilistic finality."
# Interview 2 Q8: "Show that a tamper on one node is rejected when nodes sync."
# ─────────────────────────────────────────────────────────────────────────────
class BlockchainNode:
    DIFFICULTY = 2
    def __init__(self, node_id, ministry, is_primary=False):
        self.node_id = node_id; self.ministry = ministry
        self.is_primary = is_primary; self.chain: List[Block] = []
        self._genesis()

    def _genesis(self):
        g = ZTALogEntry(
            event_type="SYSTEM_INIT", user_id="SYSTEM", agency="B-ZTA_ROOT",
            resource="GENESIS", ip_address="0.0.0.0",
            geo_location="Bangladesh Internal", device_trust_score=1.0,
            data_classification="PUBLIC", risk_score=0.0, risk_level="MINIMAL",
            risk_factors="NONE", decision="ALLOW", zt_verdict="PASS", zt_deny_reason="",
            timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
            session_id="GENESIS-000")
        blk = Block(0, g, "0"*64, "GENESIS_SIG", "SYSTEM")
        blk.mine(self.DIFFICULTY); self.chain.append(blk)

    def head_hash(self): return self.chain[-1].hash

    def validate_candidate(self, candidate: Block, keys: ECDSAKeyRegistry) -> Tuple[bool, str]:
        if candidate.previous_hash != self.head_hash(): return False, "CHAIN_LINK_MISMATCH"
        if candidate.hash != candidate._compute_hash(): return False, "HASH_RECOMPUTE_FAILED"
        eb = json.dumps(asdict(candidate.log_entry), sort_keys=True).encode()
        if not keys.verify(candidate.signer_id, eb, candidate.signature):
            return False, "ECDSA_SIGNATURE_INVALID"
        return True, "PREPARE_ACK"

    def commit(self, block: Block): self.chain.append(block)

    def validate_full_chain(self) -> Tuple[bool, List[int]]:
        tampered = []
        for i in range(1, len(self.chain)):
            c, p = self.chain[i], self.chain[i-1]
            if c.hash != c._compute_hash(): tampered.append(i); continue
            if c.previous_hash != p.hash:   tampered.append(i)
        return len(tampered) == 0, tampered

    def diverges_from(self, other: "BlockchainNode") -> List[int]:
        """Block indices where this node's recomputed hash differs from the
        hash the other node stored, i.e. where the two copies disagree."""
        n = min(len(self.chain), len(other.chain))
        return [i for i in range(n) if self.chain[i]._compute_hash() != other.chain[i].hash]

    # [C7] Gap-attack detection
    def detect_heartbeat_gaps(self, interval=7) -> List[int]:
        """
        Interview 1 Q6: 'The attacker disabled logging for 22 minutes — no
        tampered records, just no records. Completeness requires heartbeat
        anchors that reveal gaps if missing.'
        Returns the chain index at which a run of regular blocks exceeded
        `interval` without a heartbeat anchor.
        """
        gaps, run = [], 0
        for b in self.chain[1:]:
            if b.log_entry.event_type == "SYSTEM_HEARTBEAT":
                run = 0
            else:
                run += 1
                if run == interval + 1: gaps.append(b.index)
        return gaps


class PBFTConsensus:
    """
    Simplified PBFT-style flow with two nodes: PRE-PREPARE -> PREPARE -> COMMIT.
    The replica validates every candidate block before both nodes commit.
    Each node receives its own copy of the block, so a later change on one
    node does not silently change the other.
    Note: n=2 tolerates no Byzantine faults (real PBFT needs n >= 3f+1).
    """
    HEARTBEAT_EVERY = 7  # [C7]

    def __init__(self, primary: BlockchainNode, replica: BlockchainNode, keys: ECDSAKeyRegistry):
        self.primary = primary; self.replica = replica; self.keys = keys
        self._user_blocks = 0

    def _build(self, entry, sig, signer):
        c = Block(len(self.primary.chain), entry, self.primary.head_hash(), sig, signer)
        c.mine(BlockchainNode.DIFFICULTY); return c

    def _commit_both(self, block: Block):
        self.primary.commit(block)
        self.replica.commit(copy.deepcopy(block))  # independent copy per node

    def add_block(self, entry, sig, signer, verbose=False) -> Tuple[Optional[Block], List[str]]:
        log = []
        candidate = self._build(entry, sig, signer)
        log.append(f"PRE-PREPARE [{self.primary.node_id}->{self.replica.node_id}]"
                   f"  Block #{candidate.index}  {candidate.hash[:14]}...")

        ok, reason = self.replica.validate_candidate(candidate, self.keys)
        log.append(f"PREPARE     [{self.replica.node_id}->{self.primary.node_id}]"
                   f"  {'OK ACK' if ok else f'NACK ({reason})'}")
        if not ok:
            log.append("COMMIT      [ABORTED]")
            if verbose: [print(f"    {DIM}{l}{RESET}") for l in log]
            return None, log

        self._commit_both(candidate)
        log.append(f"COMMIT      [BOTH NODES OK]  Block #{candidate.index} finalised")
        if verbose: [print(f"    {DIM}{l}{RESET}") for l in log]; print()

        self._user_blocks += 1
        if self._user_blocks % self.HEARTBEAT_EVERY == 0: self._heartbeat(verbose)
        return candidate, log

    def _heartbeat(self, verbose=False):
        hb = ZTALogEntry(
            event_type="SYSTEM_HEARTBEAT", user_id="SYSTEM",
            agency="B-ZTA_MONITOR", resource="HEARTBEAT",
            ip_address="0.0.0.0", geo_location="Bangladesh Internal",
            device_trust_score=1.0, data_classification="PUBLIC",
            risk_score=0.0, risk_level="MINIMAL", risk_factors="NONE",
            decision="ALLOW", zt_verdict="PASS", zt_deny_reason="",
            timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
            session_id=f"HB-{self._user_blocks:04d}")
        sig = self.keys.sign("SYSTEM", json.dumps(asdict(hb), sort_keys=True).encode())
        c = self._build(hb, sig, "SYSTEM")
        self._commit_both(c)
        if verbose: print(f"    {DIM}[HEARTBEAT] Block #{c.index} inserted{RESET}\n")

## 5. On-chain policy management [B3]

The agency segmentation policy is itself stored on-chain. Each change needs consensus approval and is applied only after it has been committed, so the policy that controls access is as auditable as the access log.

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# [B3] ON-CHAIN POLICY MANAGEMENT
# Interview 2 Q4: "You have an immutable log for access events but a fully
# mutable and unaudited policy layer controlling those events — a significant
# architectural inconsistency."
# ─────────────────────────────────────────────────────────────────────────────
class OnChainPolicyManager:
    INITIAL_POLICY: Dict[str, List[str]] = {
        "USR-MoF-001":  ["Ministry of Finance"],
        "USR-NID-007":  ["National ID Wing"],
        "USR-BB-003":   ["Bangladesh Bank"],
        "USR-EC-002":   ["Election Commission"],
        "USR-MoH-009":  ["Ministry of Health"],
        "USR-BCC-011":  ["Bangladesh Computer Council"],
        "ADM-SYS-001":  ["Ministry of ICT", "Bangladesh Computer Council"],
        "USR-INS-007":  ["Ministry of Health"],   # insider persona [C1]
        "USR-STLN-005": ["Ministry of Finance"],  # stolen-cred persona [C1]
    }

    def __init__(self, pbft: PBFTConsensus, keys: ECDSAKeyRegistry):
        self.pbft = pbft; self.keys = keys
        self._policy = dict(self.INITIAL_POLICY)
        self.init_block: Optional[int] = None
        self._write_init()

    def _write_init(self):
        entry = ZTALogEntry(
            event_type="POLICY_INIT", user_id="SYSTEM", agency="B-ZTA_ROOT",
            resource="SEGMENTATION_POLICY", ip_address="0.0.0.0",
            geo_location="Bangladesh Internal", device_trust_score=1.0,
            data_classification="INTERNAL", risk_score=0.0, risk_level="MINIMAL",
            risk_factors="NONE", decision="ALLOW", zt_verdict="PASS", zt_deny_reason="",
            timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
            session_id="POLICY-INIT-001",
            policy_data=json.dumps(self._policy, sort_keys=True))
        sig = self.keys.sign("SYSTEM", json.dumps(asdict(entry), sort_keys=True).encode())
        blk, _ = self.pbft.add_block(entry, sig, "SYSTEM")
        if blk: self.init_block = blk.index

    def update(self, authoriser, target_uid, new_agencies, verbose=True) -> Tuple[bool, str]:
        old = self._policy.get(target_uid, [])
        entry = ZTALogEntry(
            event_type="POLICY_UPDATE", user_id=authoriser, agency="B-ZTA_ROOT",
            resource="SEGMENTATION_POLICY", ip_address="0.0.0.0",
            geo_location="Bangladesh Internal", device_trust_score=1.0,
            data_classification="INTERNAL", risk_score=1.5, risk_level="LOW",
            risk_factors="POLICY_CHANGE_PBFT_ENDORSED", decision="ALLOW",
            zt_verdict="PASS", zt_deny_reason="",
            timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
            session_id=f"POL-UPD-{random.randint(1000,9999)}",
            policy_data=json.dumps({"authoriser": authoriser, "target": target_uid,
                                    "old": old, "new": new_agencies}, sort_keys=True))
        sig = self.keys.sign("SYSTEM", json.dumps(asdict(entry), sort_keys=True).encode())
        blk, _ = self.pbft.add_block(entry, sig, "SYSTEM", verbose=verbose)
        if blk:
            self._policy[target_uid] = new_agencies  # apply only after commit
            return True, f"Policy update committed at Block #{blk.index}"
        return False, "Policy update rejected by consensus -- policy unchanged"

    def get_permitted(self, uid): return self._policy.get(uid)

## 6. Just-in-time access [C3]

Privileged accounts have no standing access. An admin request must carry a time-limited token (5 minutes here) tied to a specific user and resource, and the grant itself is logged on-chain.

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# [C3] JUST-IN-TIME ACCESS PROVISIONING
# Interview 1 Q3: "Privileged accounts require JIT access — no standing
# permissions; request time-limited elevated access through approval workflow."
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class JITToken:
    token_id: str; user_id: str; resource: str; agency: str
    granted_at: float; expires_at: float; approver_id: str
    def is_valid(self): return time.time() < self.expires_at
    def ttl(self): return max(0.0, self.expires_at - time.time())

class JITAccessManager:
    WINDOW_SEC = 300
    def __init__(self, pbft: PBFTConsensus, keys: ECDSAKeyRegistry):
        self.pbft = pbft; self.keys = keys; self._tokens: Dict[str, JITToken] = {}

    def grant(self, uid, resource, agency, approver="ADM-SYS-001") -> Tuple[str, JITToken]:
        t = JITToken(f"JIT-{random.randint(100000,999999)}", uid, resource, agency,
                     time.time(), time.time()+self.WINDOW_SEC, approver)
        self._tokens[t.token_id] = t
        entry = ZTALogEntry(
            event_type="JIT_GRANT", user_id=approver, agency="B-ZTA_ROOT",
            resource=resource, ip_address="0.0.0.0",
            geo_location="Bangladesh Internal", device_trust_score=1.0,
            data_classification="INTERNAL", risk_score=1.0, risk_level="LOW",
            risk_factors="JIT_PROVISIONING_NO_STANDING_PRIVILEGE",
            decision="ALLOW", zt_verdict="PASS", zt_deny_reason="",
            timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
            session_id=t.token_id,
            jit_token=f"{t.token_id}:window={self.WINDOW_SEC}s:approver={approver}")
        sig = self.keys.sign("SYSTEM", json.dumps(asdict(entry), sort_keys=True).encode())
        self.pbft.add_block(entry, sig, "SYSTEM")
        return t.token_id, t

    def validate(self, token_id, uid, resource) -> Tuple[bool, str]:
        t = self._tokens.get(token_id)
        if not t:                  return False, "JIT_TOKEN_NOT_FOUND"
        if t.user_id != uid:       return False, "JIT_USER_MISMATCH"
        if t.resource != resource: return False, "JIT_RESOURCE_MISMATCH"
        if not t.is_valid():       return False, "JIT_TOKEN_EXPIRED"
        return True, f"JIT_VALID ({t.ttl():.0f}s remaining)"

## 7. Step-up authentication [C2]

When a request needs extra assurance, the user must pass a secondary challenge. Users without an enrolled secondary method are denied rather than passed through.

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# [C2] STEP-UP AUTHENTICATION ENFORCEMENT
# Interview 1 Q3: "STEP_UP_AUTH is a label with no enforcement mechanism —
# a user who triggers it simply passes through as if nothing happened."
# ─────────────────────────────────────────────────────────────────────────────
class StepUpAuthManager:
    REGISTRY: Dict[str, str] = {
        "USR-MoF-001": "SUPERVISOR_OTP",    "USR-NID-007": "HARDWARE_KEY_2",
        "USR-BB-003":  "SUPERVISOR_OTP",    "USR-EC-002":  "HARDWARE_KEY_2",
        "USR-MoH-009": "SUPERVISOR_OTP",    "USR-BCC-011": "SMS_OTP",
        "ADM-SYS-001": "DUAL_APPROVAL",     "USR-STLN-005":"SUPERVISOR_OTP",
        # USR-INS-007 deliberately NOT enrolled -> step-up fails -> DENY [C1]
    }
    def challenge(self, uid) -> Tuple[bool, str]:
        method = self.REGISTRY.get(uid)
        if not method:
            return False, f"STEP_UP_FAILED:no_secondary_method_for_{uid}->DENY"
        return True, f"STEP_UP_PASSED:{method}_accepted"

## 8. Risk engine [C5, C6]

Ten weighted factors produce a 0 to 10 risk score. Two behavioural signals catch threats that simple rate limits miss: **semantic velocity** (unique resources touched per minute) and a **reputation EMA** that weights recent behaviour, so a long-trusted user who turns malicious is not masked by their history.

A weighted average can dilute a single severe indicator, so two indicators, mass exfiltration and semantic velocity, **force step-up authentication** even when the overall score is only moderate.

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# ENHANCED RISK ENGINE
# [C5] Semantic velocity  — unique resources per 60s window (UEBA)
# [C6] Reputation EMA     — exponential moving avg, recent-behaviour-weighted
# Interview 1 Q4: "A user querying citizen_nid_db once per minute for 8 hours
# is not velocity-anomalous but is semantically very suspicious."
# Interview 1 Q4: "A long-standing trusted user who turns malicious starts
# with a low reputation factor that masks the current anomaly."
# ─────────────────────────────────────────────────────────────────────────────
class RiskEngine:
    WEIGHTS = {
        "data_classification": 2.5, "geo_risk":          2.0,
        "device_trust":        1.5, "time_of_access":    1.0,
        "data_volume":         1.0, "semantic_velocity": 0.8,
        "reputation":          0.7, "lateral_movement":  0.5,
        "failed_attempts":     0.5, "privilege_level":   0.5,
    }
    MAX_RAW = sum(WEIGHTS.values())
    EMA_ALPHA = 0.35
    # Severe indicators that force step-up regardless of the weighted score
    ESCALATION_FLAGS = ("MASS_EXFIL", "SEMANTIC_VELOCITY")

    def __init__(self):
        self.hist: Dict = defaultdict(lambda: {
            "res_window": [], "agencies": set(), "failed_auths": 0,
            "ema_risk": 0.0,  "last_active": 0.0, "count": 0,
        })

    def _semantic_vel(self, uid, resource) -> Tuple[float, List[str]]:
        h = self.hist[uid]; now = time.time()
        h["res_window"].append((resource, now))
        h["res_window"] = [(r,t) for r,t in h["res_window"] if now-t < 60]
        unique = len(set(r for r,_ in h["res_window"]))
        if unique >= 6: return 1.0, [f"SEMANTIC_VELOCITY({unique}_unique/min)"]
        if unique >= 3: return 0.5, [f"ELEVATED_UNIQUE_RES({unique}/min)"]
        return 0.0, []

    def _reputation(self, uid, raw_score) -> Tuple[float, List[str]]:
        h = self.hist[uid]; now = time.time()
        if h["last_active"] > 0 and (now - h["last_active"]) > 3600:
            h["ema_risk"] *= 0.5   # decay toward neutral on inactivity
        h["ema_risk"] = self.EMA_ALPHA*raw_score + (1-self.EMA_ALPHA)*h["ema_risk"]
        h["last_active"] = now
        rep = min(1.0, h["ema_risk"]/10.0)
        flags = ([f"POOR_REPUTATION_EMA({h['ema_risk']:.1f})"] if h["ema_risk"] > 6.5 else [])
        return rep, flags

    def score(self, uid, resource, ip, trust, hour, vol_kb, is_admin=False):
        h = self.hist[uid]; fac: Dict[str, float] = {}; flags: List[str] = []

        _, cls = DATA_CLASS.get(resource, ("INTERNAL",2))
        fac["data_classification"] = (cls-1)/4.0
        if cls >= 4: flags.append(f"HIGH_SENSITIVITY({resource})")

        geo_label, geo_score = geo_risk(ip)
        fac["geo_risk"] = geo_score/5.0
        if geo_score >= 3: flags.append(f"SUSPICIOUS_GEO({geo_label})")

        fac["device_trust"] = max(0, 1.0-trust)
        if trust < 0.50: flags.append(f"LOW_DEVICE_TRUST({trust:.3f})")

        if 0 <= hour < 5:     fac["time_of_access"]=1.0; flags.append("DEAD_OF_NIGHT")
        elif hour in (5,6,21,22,23): fac["time_of_access"]=0.6; flags.append("OFF_HOURS")
        else:                 fac["time_of_access"]=0.0

        if vol_kb > 10000:    fac["data_volume"]=1.0; flags.append(f"MASS_EXFIL({vol_kb:.0f}KB)")
        elif vol_kb > 1000:   fac["data_volume"]=0.6; flags.append(f"LARGE_EXPORT({vol_kb:.0f}KB)")
        elif vol_kb > 300:    fac["data_volume"]=0.3
        else:                 fac["data_volume"]=0.0

        sv, sv_fl = self._semantic_vel(uid, resource)
        fac["semantic_velocity"] = sv; flags.extend(sv_fl)

        partial = sum(self.WEIGHTS[k]*v for k,v in fac.items()
                      if k not in ("semantic_velocity","reputation"))
        rep, rep_fl = self._reputation(uid, (partial/self.MAX_RAW)*10)
        fac["reputation"] = rep; flags.extend(rep_fl)

        agencies = len(h["agencies"])
        if agencies >= 4:     fac["lateral_movement"]=1.0; flags.append(f"LATERAL({agencies}_ag)")
        elif agencies >= 2:   fac["lateral_movement"]=0.4
        else:                 fac["lateral_movement"]=0.0

        failed = h["failed_auths"]
        if failed >= 3:       fac["failed_attempts"]=1.0; flags.append(f"REPEAT_FAIL({failed})")
        elif failed >= 1:     fac["failed_attempts"]=0.4
        else:                 fac["failed_attempts"]=0.0

        fac["privilege_level"] = 0.8 if is_admin else 0.0
        if is_admin: flags.append("PRIVILEGED_ADMIN")

        raw = sum(self.WEIGHTS[k]*v for k,v in fac.items())
        score = round((raw/self.MAX_RAW)*10.0, 2)
        level = ("CRITICAL" if score>=7.5 else "HIGH" if score>=5.5
                 else "MEDIUM" if score>=3.5 else "LOW" if score>=1.5 else "MINIMAL")
        h["count"] += 1
        return score, level, flags

    def record_agency(self, uid, ag): self.hist[uid]["agencies"].add(ag)
    def record_fail(self, uid):       self.hist[uid]["failed_auths"] += 1

    def make_decision(self, score, level, trust, flags=()):
        if level == "CRITICAL":                      return "DENY"
        if level == "HIGH" and trust < 0.55:         return "DENY"
        if level == "HIGH":                          return "STEP_UP_AUTH"
        if any(f.startswith(self.ESCALATION_FLAGS) for f in flags):
            return "STEP_UP_AUTH"                    # severe indicator overrides the average
        if level == "MEDIUM" and trust < 0.40:       return "STEP_UP_AUTH"
        return "ALLOW"

## 9. Zero Trust gateway [C8]

The gateway (policy decision point) checks MFA, device posture and on-chain segmentation before the risk engine runs. It is kept logically separate from the audit logger, following NIST SP 800-207's warning that a compromised decision point should not also be able to silence the audit trail. Soft MFA (TOTP or SMS) is rejected from high-risk locations, where a hardware key is required.

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# [C8] ZTA GATEWAY — CONTROL-PLANE SEPARATION
# Interview 1 Q5: "The PDP and logging must be independently operated —
# otherwise a compromise of the PDP silently disables both enforcement and
# audit. NIST SP 800-207 explicitly warns against this circular dependency."
# ─────────────────────────────────────────────────────────────────────────────
class ZeroTrustGateway:
    MFA_REGISTRY: Dict[str, str] = {
        "USR-MoF-001": "HARDWARE_KEY", "USR-NID-007": "TOTP",
        "USR-BB-003":  "HARDWARE_KEY", "USR-EC-002":  "TOTP",
        "USR-MoH-009": "TOTP",         "USR-BCC-011": "SMS",
        "ADM-SYS-001": "HARDWARE_KEY", "USR-INS-007": "TOTP",
        "USR-STLN-005":"TOTP",
        # ATCK-EXT-001 NOT registered -> MFA BLOCK
    }
    DEVICE_TRUST_THRESHOLD = 0.70

    def __init__(self, policy: OnChainPolicyManager):
        self.policy = policy  # [B3] on-chain source of truth

    def _mfa(self, uid, ip):
        method = self.MFA_REGISTRY.get(uid, "NONE")
        if method == "NONE": return False, "NONE", "No MFA token registered"
        geo_label, geo_score = geo_risk(ip)
        if geo_score >= 3 and method != "HARDWARE_KEY":
            return False, method, f"Soft MFA insufficient from high-risk geo ({geo_label})"
        return True, method, f"{method} challenge accepted"

    def _device(self, uid, trust):
        issues = []
        if uid not in self.MFA_REGISTRY:
            issues += ["UNRECOGNISED_DEVICE","NO_MDM_ENROLMENT","NO_CORP_CERT"]
        if trust < self.DEVICE_TRUST_THRESHOLD:
            issues.append(f"TRUST_BELOW_THRESHOLD({trust:.3f})")
        if trust < 0.50: issues.append("ENDPOINT_NOT_COMPLIANT")
        return len(issues) == 0, issues

    def _segment(self, uid, agency):
        allowed = self.policy.get_permitted(uid)
        if allowed is None: return False, "USER_NOT_IN_ON_CHAIN_POLICY"
        if agency not in allowed: return False, f"CROSS_AGENCY_VIOLATION({agency})"
        return True, f"Access to {agency} permitted by on-chain policy [B3]"

    def evaluate(self, uid, agency, resource, ip, trust) -> ZTAIdentityCheck:
        mfa_ok, mfa_method, mfa_detail = self._mfa(uid, ip)
        dev_ok, dev_issues             = self._device(uid, trust)
        seg_ok, seg_detail             = self._segment(uid, agency)

        if not mfa_ok:   verdict, deny = "BLOCK",     f"MFA_FAILED — {mfa_detail}"
        elif not seg_ok: verdict, deny = "BLOCK",     f"SEGMENTATION — {seg_detail}"
        elif not dev_ok: verdict, deny = "CHALLENGE", f"DEVICE — {'; '.join(dev_issues)}"
        else:            verdict, deny = "PASS",      ""

        return ZTAIdentityCheck(
            user_id=uid, agency=agency, resource=resource,
            mfa_passed=mfa_ok, mfa_method=mfa_method, mfa_detail=mfa_detail,
            device_compliant=dev_ok, device_issues=dev_issues,
            segmentation_allowed=seg_ok, seg_detail=seg_detail,
            zt_verdict=verdict, deny_reason=deny)

## 10. Integrated policy engine

Combines the JIT, gateway, risk and step-up checks into one decision, then signs the resulting log entry with the requester's key and submits it for consensus. Admin requests without a valid JIT token have their trust set to zero, so the gateway challenges them.

In [11]:
class BZTAPolicyEngine:
    def __init__(self, pbft, risk, gateway, keys, step_up, jit):
        self.pbft=pbft; self.risk=risk; self.gateway=gateway
        self.keys=keys; self.step_up=step_up; self.jit=jit

    def process(self, uid, agency, resource, ip, base_trust, hour,
                vol_kb=0.0, is_admin=False, jit_token="",
                elapsed_min=0.0, anomalies=0, persona="",
                pbft_verbose=False) -> Tuple[Optional[Block], ZTAIdentityCheck]:

        trust = (decay_trust(base_trust, elapsed_min, anomalies)
                 if elapsed_min > 0 else base_trust)

        if is_admin:  # [C3] no standing admin privilege: a valid JIT token is required
            jit_ok, jit_msg = (self.jit.validate(jit_token, uid, resource)
                               if jit_token else (False, "JIT_TOKEN_MISSING"))
            if not jit_ok: trust = 0.0   # force device-compliance challenge

        zt = self.gateway.evaluate(uid, agency, resource, ip, trust)
        self.risk.record_agency(uid, agency)
        score, level, flags = self.risk.score(uid, resource, ip, trust, hour, vol_kb, is_admin)

        step_up_result = ""
        if zt.zt_verdict == "BLOCK":
            decision = "ZTA_BLOCK"
            flags.append(f"ZTA_GATE:{zt.deny_reason[:50]}")
            self.risk.record_fail(uid)
        elif zt.zt_verdict == "CHALLENGE":
            decision = "ZTA_CHALLENGE"
            flags.append("ZTA_DEVICE_CHALLENGE")
        else:
            decision = self.risk.make_decision(score, level, trust, flags)
            if decision == "STEP_UP_AUTH":
                su_ok, su_msg = self.step_up.challenge(uid)
                step_up_result = su_msg
                decision = "ALLOW" if su_ok else "DENY"
                flags.append(su_msg.split(":")[0])
                if not su_ok: self.risk.record_fail(uid)
            elif decision == "DENY":
                self.risk.record_fail(uid)

        cls_label, _ = DATA_CLASS.get(resource, ("INTERNAL", 2))
        geo_label, _ = geo_risk(ip)

        entry = ZTALogEntry(
            event_type="AUTH_REQUEST" if vol_kb < 500 else "DATA_EXPORT",
            user_id=uid, agency=agency, resource=resource,
            ip_address=ip, geo_location=geo_label,
            device_trust_score=round(trust, 3),
            data_classification=cls_label,
            risk_score=score, risk_level=level,
            risk_factors="|".join(flags) if flags else "NONE",
            decision=decision, zt_verdict=zt.zt_verdict,
            zt_deny_reason=zt.deny_reason,
            timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
            session_id=f"SID-{random.randint(100000,999999)}",
            data_volume_kb=vol_kb,
            anomaly_flags="|".join(f for f in flags if any(k in f for k in
                ["EXFIL","VELOCITY","LATERAL","ZTA","STEP_UP","SEMANTIC",
                 "REPUTATION","NIGHT","HOURS","TRUST"])),
            jit_token=jit_token, step_up_result=step_up_result, threat_persona=persona)

        eb     = json.dumps(asdict(entry), sort_keys=True).encode()
        signer = uid if self.keys.has_key(uid) else "SYSTEM"
        sig    = self.keys.sign(signer, eb)
        block, _ = self.pbft.add_block(entry, sig, signer, verbose=pbft_verbose)
        return block, zt

## 11. Display helpers

In [12]:
def bar(score, w=20):
    filled = int((score/10.0)*w)
    color  = GREEN if score < 3.5 else (YELLOW if score < 5.5 else RED)
    return f"{color}{'|'*filled}{'-'*(w-filled)}{RESET} {score:.1f}/10"

def sep(title="", char="=", width=72):
    if title:
        pad = (width-len(title)-2)//2
        print(f"\n{BOLD}{CYAN}{char*pad} {title} {char*(width-pad-len(title)-2)}{RESET}")
    else:
        print(f"\n{BOLD}{CYAN}{char*width}{RESET}")

def section(title):
    print(f"\n{BOLD}{YELLOW}+-- {title} {'-'*max(0, 60-len(title))}+{RESET}")

def print_zt(zt: ZTAIdentityCheck):
    vc = GREEN if zt.zt_verdict=="PASS" else (YELLOW if zt.zt_verdict=="CHALLENGE" else RED)
    mc = GREEN if zt.mfa_passed else RED
    dc = GREEN if zt.device_compliant else (YELLOW if zt.zt_verdict=="CHALLENGE" else RED)
    sc = GREEN if zt.segmentation_allowed else RED
    print(f"  {CYAN}ZTA Gateway -- {BOLD}{zt.user_id}{RESET}")
    print(f"    MFA     : {mc}{'OK' if zt.mfa_passed else 'FAIL'} {zt.mfa_method}{RESET}  ({zt.mfa_detail})")
    ds = "COMPLIANT" if zt.device_compliant else f"NON-COMPLIANT -- {', '.join(zt.device_issues)}"
    print(f"    Device  : {dc}{'OK' if zt.device_compliant else 'FAIL'} {ds}{RESET}")
    print(f"    Segment : {sc}{'OK' if zt.segmentation_allowed else 'FAIL'} {zt.seg_detail}{RESET}")
    print(f"    Verdict : {vc}{BOLD}{zt.zt_verdict}{RESET}"
          + (f"  <- {RED}{zt.deny_reason}{RESET}" if zt.deny_reason else ""))
    print()

def print_block(blk: Block, zt: ZTAIdentityCheck, tag=""):
    e   = blk.log_entry
    dc  = (GREEN if e.decision=="ALLOW" else YELLOW if "CHALLENGE" in e.decision else RED)
    cc  = CLASS_COLOR.get(e.data_classification, WHITE)
    vc  = GREEN if zt.zt_verdict=="PASS" else (YELLOW if zt.zt_verdict=="CHALLENGE" else RED)
    ts  = f" {RED}[{tag}]{RESET}" if tag else ""
    sig_ok = blk.signature and "NO_ECDSA" not in blk.signature
    sc  = GREEN if sig_ok else YELLOW
    print(f"  {CYAN}Block #{blk.index:03d}{RESET}{ts}  {DIM}{blk.hash[:18]}...{RESET}  {sc}[{'SIGNED' if sig_ok else 'NO-SIG'}]{RESET}")
    print(f"  {'-'*68}")
    print(f"  User    : {BOLD}{e.user_id:<22}{RESET}  Agency  : {e.agency}")
    print(f"  Resource: {e.resource:<22}  Class   : {cc}{e.data_classification}{RESET}")
    print(f"  IP      : {e.ip_address:<22}  Geo     : {e.geo_location}")
    print(f"  Trust   : {e.device_trust_score:<22.3f}  Decision: {dc}{BOLD}{e.decision}{RESET}")
    print(f"  ZT Gate : {vc}{BOLD}{zt.zt_verdict:<10}{RESET}  Risk    : {bar(e.risk_score)}")
    if e.step_up_result:
        sc2 = GREEN if "PASSED" in e.step_up_result else RED
        print(f"  StepUp  : {sc2}{e.step_up_result}{RESET}")
    if e.jit_token:
        print(f"  JIT     : {CYAN}{e.jit_token[:65]}{RESET}")
    if e.threat_persona:
        print(f"  Persona : {MAGENTA}[{e.threat_persona}]{RESET}")
    if e.risk_factors != "NONE":
        fl = e.risk_factors.split("|")
        print(f"  Flags   : {RED}{', '.join(fl[:3])}{RESET}",
              end=f"  +{len(fl)-3} more\n" if len(fl)>3 else "\n")
    if e.data_volume_kb > 0:
        print(f"  Volume  : {BOLD}{e.data_volume_kb:>10.0f} KB{RESET}  Event: {e.event_type}")

## 12. Simulation

### System initialisation
Registers keys, starts both nodes from the same genesis block (each with its own copy) and writes the initial policy on-chain.

In [13]:
sep("B-ZTA v4.1 -- Zero Trust + Blockchain Audit Simulation")
print(f"  {DIM}Enhancements: ECDSA [B1] | Dual-node consensus [B2] | On-chain policy [B3]{RESET}")
print(f"  {DIM}JIT access [C3] | Step-up enforced [C2] | Trust decay [C4]{RESET}")
print(f"  {DIM}Heartbeat gaps [C7] | Semantic velocity [C5] | Rep EMA [C6]{RESET}")

section("SYSTEM INIT -- ECDSA Keys + Consensus Nodes + On-Chain Policy")

keys = ECDSAKeyRegistry()
all_users = ["USR-MoF-001","USR-NID-007","USR-BB-003","USR-EC-002",
             "USR-MoH-009","USR-BCC-011","ADM-SYS-001",
             "USR-INS-007","USR-STLN-005","SYSTEM"]
for u in all_users: keys.register(u)
print(f"  {GREEN}OK ECDSA SECP256R1 keys registered for {len(all_users)} identities{RESET}")
print(f"  {DIM}  [B4] Post-quantum forward path (not implemented): ML-DSA / CRYSTALS-Dilithium (NIST FIPS 204){RESET}")
for u in all_users[:4]:
    print(f"    {u:<22}  pubkey: {CYAN}{keys.pubkey_short(u)}{RESET}")
print(f"    ... and {len(all_users)-4} more\n")

node_a = BlockchainNode("NODE-A", "Bangladesh Computer Council", is_primary=True)
node_b = BlockchainNode("NODE-B", "Ministry of ICT",             is_primary=False)
# Both nodes must start from the same genesis root so chain links match.
# Node-B gets its own copy, so later changes to Node-A cannot affect it.
# In production each node would verify the genesis block out of band
# (e.g. a signed genesis certificate from the ordering service).
node_b.chain = copy.deepcopy(node_a.chain)
pbft   = PBFTConsensus(node_a, node_b, keys)
print(f"  {GREEN}OK Consensus nodes: {node_a.node_id} (primary) <-> {node_b.node_id} (replica){RESET}")

policy  = OnChainPolicyManager(pbft, keys)
print(f"  {GREEN}OK POLICY_INIT committed at Block #{policy.init_block} -- policy on-chain [B3]{RESET}")

gateway  = ZeroTrustGateway(policy)       # [C8] separate from audit
risk_eng = RiskEngine()
step_up  = StepUpAuthManager()
jit_mgr  = JITAccessManager(pbft, keys)
engine   = BZTAPolicyEngine(pbft, risk_eng, gateway, keys, step_up, jit_mgr)


======== B-ZTA v4.1 -- Zero Trust + Blockchain Audit Simulation ========
  Enhancements: ECDSA [B1] | Dual-node consensus [B2] | On-chain policy [B3]
  JIT access [C3] | Step-up enforced [C2] | Trust decay [C4]
  Heartbeat gaps [C7] | Semantic velocity [C5] | Rep EMA [C6]

+-- SYSTEM INIT -- ECDSA Keys + Consensus Nodes + On-Chain Policy +
  OK ECDSA SECP256R1 keys registered for 10 identities
    [B4] Post-quantum forward path (not implemented): ML-DSA / CRYSTALS-Dilithium (NIST FIPS 204)
    USR-MoF-001             pubkey: 025d3e6a4809682e...
    USR-NID-007             pubkey: 03b719c71910b001...
    USR-BB-003              pubkey: 03b6489829dd6930...
    USR-EC-002              pubkey: 02fbba26fed3ec8b...
    ... and 6 more

  OK Consensus nodes: NODE-A (primary) <-> NODE-B (replica)
  OK POLICY_INIT committed at Block #1 -- policy on-chain [B3]


### Phase 0: Gateway pre-checks
Identity, device and segmentation checks only, before any request is logged.

In [14]:
section("PHASE 0: ZTA Gateway -- Identity + Device + On-Chain Segmentation")
print(f"  {DIM}Segmentation policy read from on-chain store [B3].{RESET}\n")

p0 = [
    ("USR-MoF-001",  "Ministry of Finance",      "tax_records",    "10.0.1.45",  0.92),
    ("USR-NID-007",  "National ID Wing",         "citizen_nid_db", "10.0.2.12",  0.91),
    ("ADM-SYS-001",  "Ministry of ICT",          "passport_db",    "172.16.1.1", 0.96),
    ("USR-INS-007",  "Ministry of Health",       "health_records", "10.0.5.55",  0.85),
    ("ATCK-EXT-001", "Ministry of Home Affairs", "staff_directory","103.0.10.5", 0.55),
    ("ATCK-EXT-001", "Election Commission",      "voter_registry", "203.0.113.5",0.41),
]
p0_pass = p0_block = 0
for uid, ag, res, ip, tr in p0:
    zt = gateway.evaluate(uid, ag, res, ip, tr)
    print_zt(zt)
    if zt.zt_verdict == "PASS": p0_pass += 1
    else: p0_block += 1
print(f"  Phase 0: {GREEN}{p0_pass} PASS{RESET}  {RED}{p0_block} BLOCKED{RESET}"
      f" -- external attacker stopped at the gateway")


+-- PHASE 0: ZTA Gateway -- Identity + Device + On-Chain Segmentation +
  Segmentation policy read from on-chain store [B3].

  ZTA Gateway -- USR-MoF-001
    MFA     : OK HARDWARE_KEY  (HARDWARE_KEY challenge accepted)
    Device  : OK COMPLIANT
    Segment : OK Access to Ministry of Finance permitted by on-chain policy [B3]
    Verdict : PASS

  ZTA Gateway -- USR-NID-007
    MFA     : OK TOTP  (TOTP challenge accepted)
    Device  : OK COMPLIANT
    Segment : OK Access to National ID Wing permitted by on-chain policy [B3]
    Verdict : PASS

  ZTA Gateway -- ADM-SYS-001
    MFA     : OK HARDWARE_KEY  (HARDWARE_KEY challenge accepted)
    Device  : OK COMPLIANT
    Segment : OK Access to Ministry of ICT permitted by on-chain policy [B3]
    Verdict : PASS

  ZTA Gateway -- USR-INS-007
    MFA     : OK TOTP  (TOTP challenge accepted)
    Device  : OK COMPLIANT
    Segment : OK Access to Ministry of Health permitted by on-chain policy [B3]
    Verdict : PASS

  ZTA Gateway -- ATCK-EXT

### Phase 1: Normal government operations
Six legitimate requests, each validated by the replica and committed to both nodes.

In [15]:
section("PHASE 1: Normal Government Operations -- Consensus + ECDSA Visible")
print(f"  {DIM}Each block: PRE-PREPARE -> PREPARE (ECDSA check) -> COMMIT both nodes.{RESET}\n")

legit = [
    ("USR-MoF-001","Ministry of Finance",         "tax_records",           "10.0.1.45", 0.92, 9,120.0,False),
    ("USR-NID-007","National ID Wing",            "citizen_nid_db",        "10.0.2.12", 0.91,10, 45.0,False),
    ("USR-BB-003", "Bangladesh Bank",             "financial_transactions","10.0.3.77", 0.95,11,200.0,False),
    ("USR-EC-002", "Election Commission",         "voter_registry",        "10.0.4.33", 0.87, 9, 80.0,False),
    ("USR-MoH-009","Ministry of Health",          "health_records",        "10.0.5.21", 0.90,14, 30.0,False),
    ("USR-BCC-011","Bangladesh Computer Council", "staff_directory",       "172.16.0.5",0.88,13,  5.0,False),
]
for uid,ag,res,ip,tr,hr,vol,adm in legit:
    blk,zt = engine.process(uid,ag,res,ip,tr,hr,vol,adm,pbft_verbose=True)
    print_block(blk, zt)

va,_ = node_a.validate_full_chain(); vb,_ = node_b.validate_full_chain()
ok_col = GREEN if (va and vb) else RED
print(f"\n  {ok_col}Node-A valid: {va}  Node-B valid: {vb}  Blocks: {len(node_a.chain)}{RESET}")


+-- PHASE 1: Normal Government Operations -- Consensus + ECDSA Visible +
  Each block: PRE-PREPARE -> PREPARE (ECDSA check) -> COMMIT both nodes.

    PRE-PREPARE [NODE-A->NODE-B]  Block #2  0054ccb063994c...
    PREPARE     [NODE-B->NODE-A]  OK ACK
    COMMIT      [BOTH NODES OK]  Block #2 finalised

  Block #002  0054ccb063994c98d5...  [SIGNED]
  --------------------------------------------------------------------
  User    : USR-MoF-001             Agency  : Ministry of Finance
  Resource: tax_records             Class   : CONFIDENTIAL
  IP      : 10.0.1.45               Geo     : Bangladesh Internal
  Trust   : 0.920                   Decision: ALLOW
  ZT Gate : PASS        Risk    : ||------------------ 1.3/10
  Volume  :        120 KB  Event: AUTH_REQUEST
    PRE-PREPARE [NODE-A->NODE-B]  Block #3  00d038c5718270...
    PREPARE     [NODE-B->NODE-A]  OK ACK
    COMMIT      [BOTH NODES OK]  Block #3 finalised

  Block #003  00d038c5718270598b...  [SIGNED]
  -----------------------

### Phase 2: Persona 1, external attacker [C1]
No MFA token, an unrecognised device and no entry in the on-chain policy. Every denial is still recorded on-chain for forensics.

In [16]:
section("PHASE 2: [C1] Threat Persona 1 -- External Attacker (no credentials)")
print(f"  {DIM}No MFA token | Unrecognised device | Not in on-chain policy.{RESET}")
print(f"  {DIM}All denial events recorded on-chain for forensics.{RESET}\n")

p1_attacks = [
    ("ATCK-EXT-001","Ministry of Home Affairs","staff_directory",       "103.0.10.5", 0.55,19,  2.0,False),
    ("ATCK-EXT-001","Bangladesh Bank",         "financial_transactions","103.0.10.5", 0.48,21, 20.0,False),
    ("ATCK-EXT-001","Ministry of Home Affairs","citizen_nid_db",        "203.0.113.5",0.41, 2,15000.0,True),
    ("ATCK-EXT-001","Ministry of Home Affairs","intelligence_reports",  "185.0.0.1",  0.30, 4,22000.0,True),
]
attack_blocks = []
for uid,ag,res,ip,tr,hr,vol,adm in p1_attacks:
    blk,zt = engine.process(uid,ag,res,ip,tr,hr,vol,adm,persona="PERSONA_1_EXTERNAL")
    attack_blocks.append(blk); print_block(blk,zt,"P1-EXT")


+-- PHASE 2: [C1] Threat Persona 1 -- External Attacker (no credentials) +
  No MFA token | Unrecognised device | Not in on-chain policy.
  All denial events recorded on-chain for forensics.

  Block #009 [P1-EXT]  0001e7cfc5aef9bfe1...  [SIGNED]
  --------------------------------------------------------------------
  User    : ATCK-EXT-001            Agency  : Ministry of Home Affairs
  Resource: staff_directory         Class   : INTERNAL
  IP      : 103.0.10.5              Geo     : Bangladesh ISP
  Trust   : 0.550                   Decision: ZTA_BLOCK
  ZT Gate : BLOCK       Risk    : |||----------------- 1.6/10
  Persona : [PERSONA_1_EXTERNAL]
  Flags   : ZTA_GATE:MFA_FAILED — No MFA token registered
  Volume  :          2 KB  Event: AUTH_REQUEST
  Block #010 [P1-EXT]  0049cafa1f1bb074df...  [SIGNED]
  --------------------------------------------------------------------
  User    : ATCK-EXT-001            Agency  : Bangladesh Bank
  Resource: financial_transactions  Class   : SECR

### Phase 3: Persona 2, malicious insider [C1]
A Ministry of Health user with valid MFA, a compliant device and an authorised agency sweeps through several databases. Early requests look normal. Once six unique resources are touched within a minute, semantic velocity forces step-up authentication, and because this user has no secondary method enrolled, the request is denied. The final mass export is denied the same way.

In [17]:
section("PHASE 3: [C1] Threat Persona 2 -- Malicious Insider (valid credentials)")
print(f"  {DIM}USR-INS-007: valid MFA OK  MDM device OK  authorised agency OK{RESET}")
print(f"  {DIM}[C5] 6 unique resources in 60s -> SEMANTIC_VELOCITY -> forced step-up{RESET}")
print(f"  {DIM}[C2] Step-up -> not enrolled -> DENY{RESET}\n")

insider = [
    ("USR-INS-007","Ministry of Health","health_records",        "10.0.5.55",0.85, 9, 150.0,False),
    ("USR-INS-007","Ministry of Health","staff_directory",       "10.0.5.55",0.85, 9,  50.0,False),
    ("USR-INS-007","Ministry of Health","land_registry",         "10.0.5.55",0.85,10, 200.0,False),
    ("USR-INS-007","Ministry of Health","passport_db",           "10.0.5.55",0.85,10, 400.0,False),
    ("USR-INS-007","Ministry of Health","tax_records",           "10.0.5.55",0.85,10, 600.0,False),
    ("USR-INS-007","Ministry of Health","citizen_nid_db",        "10.0.5.55",0.83,11,8500.0,False),
    ("USR-INS-007","Ministry of Health","financial_transactions", "10.0.5.55",0.80, 2,18000.0,False),
]
for uid,ag,res,ip,tr,hr,vol,adm in insider:
    blk,zt = engine.process(uid,ag,res,ip,tr,hr,vol,adm,persona="PERSONA_2_INSIDER")
    print_block(blk,zt,"P2-INS" if vol>1000 else "")


+-- PHASE 3: [C1] Threat Persona 2 -- Malicious Insider (valid credentials) +
  USR-INS-007: valid MFA OK  MDM device OK  authorised agency OK
  [C5] 6 unique resources in 60s -> SEMANTIC_VELOCITY -> forced step-up
  [C2] Step-up -> not enrolled -> DENY

  Block #013  00e7b9287763eb717c...  [SIGNED]
  --------------------------------------------------------------------
  User    : USR-INS-007             Agency  : Ministry of Health
  Resource: health_records          Class   : CONFIDENTIAL
  IP      : 10.0.5.55               Geo     : Bangladesh Internal
  Trust   : 0.850                   Decision: ALLOW
  ZT Gate : PASS        Risk    : ||------------------ 1.4/10
  Persona : [PERSONA_2_INSIDER]
  Volume  :        150 KB  Event: AUTH_REQUEST
  Block #014  003a9d4d3d8330a9f3...  [SIGNED]
  --------------------------------------------------------------------
  User    : USR-INS-007             Agency  : Ministry of Health
  Resource: staff_directory         Class   : INTERNAL
  IP   

### Phase 4: Persona 4, stolen credentials [C1]
The attacker has a Ministry of Finance user's password and TOTP code, but connects from a high-risk foreign IP. The gateway requires a hardware key from high-risk locations, so both requests are blocked at MFA before the risk engine is consulted.

In [18]:
section("PHASE 4: [C1] Threat Persona 4 -- Stolen Credentials")
print(f"  {DIM}USR-STLN-005: stolen password + TOTP | personal laptop (low trust){RESET}")
print(f"  {DIM}Foreign high-risk IP -> soft MFA not accepted -> ZTA_BLOCK{RESET}\n")

stolen = [
    ("USR-STLN-005","Ministry of Finance","tax_records",           "194.0.0.50",0.62,21,  200.0,False),
    ("USR-STLN-005","Ministry of Finance","financial_transactions","194.0.0.50",0.60,22,14000.0,False),
]
for uid,ag,res,ip,tr,hr,vol,adm in stolen:
    blk,zt = engine.process(uid,ag,res,ip,tr,hr,vol,adm,persona="PERSONA_4_STOLEN_CRED")
    print_block(blk,zt,"P4-STLN")


+-- PHASE 4: [C1] Threat Persona 4 -- Stolen Credentials --------+
  USR-STLN-005: stolen password + TOTP | personal laptop (low trust)
  Foreign high-risk IP -> soft MFA not accepted -> ZTA_BLOCK

  Block #021 [P4-STLN]  00acd994ede1257624...  [SIGNED]
  --------------------------------------------------------------------
  User    : USR-STLN-005            Agency  : Ministry of Finance
  Resource: tax_records             Class   : CONFIDENTIAL
  IP      : 194.0.0.50              Geo     : Eastern Europe
  Trust   : 0.620                   Decision: ZTA_BLOCK
  ZT Gate : BLOCK       Risk    : ||||||-------------- 3.4/10
  Persona : [PERSONA_4_STOLEN_CRED]
  Flags   : SUSPICIOUS_GEO(Eastern Europe), OFF_HOURS, ZTA_GATE:MFA_FAILED — Soft MFA insufficient from high-risk 
  Volume  :        200 KB  Event: AUTH_REQUEST
  Block #022 [P4-STLN]  0083954a65cff08337...  [SIGNED]
  --------------------------------------------------------------------
  User    : USR-STLN-005            Agency  :

### Phase 5: JIT admin access and an on-chain policy update [C3, B3]
Persona 3 (privileged admin) receives a 5-minute token for one resource. Then a policy change is proposed, endorsed by both nodes and applied.

In [19]:
section("PHASE 5: [C3] JIT Access + [B3] On-Chain Policy Update")

print(f"\n  {BOLD}[C3] JIT token grant -- ADM-SYS-001 needs time-limited elevated access:{RESET}")
tid,tok = jit_mgr.grant("ADM-SYS-001","military_personnel_db",
                         "Bangladesh Computer Council","ADM-SYS-001")
print(f"  {GREEN}OK JIT token: {tid}  expires in {tok.ttl():.0f}s{RESET}")
print(f"  {DIM}  Grant event on-chain -- no standing admin privilege exists [C3].{RESET}\n")

admin_blk,zt = engine.process("ADM-SYS-001","Bangladesh Computer Council",
                         "military_personnel_db","172.16.1.1",0.96,10,55.0,
                         is_admin=True,jit_token=tid,
                         persona="PERSONA_3_PRIVILEGED_ADMIN",pbft_verbose=True)
print_block(admin_blk,zt,"JIT-ADMIN")

print(f"\n  {BOLD}[B3] On-chain policy update -- USR-EC-002 expanded to National ID Wing:{RESET}")
print(f"  {DIM}  Requires endorsement from both nodes. Every change permanently auditable.{RESET}\n")
policy_ok,msg = policy.update("ADM-SYS-001","USR-EC-002",
                        ["Election Commission","National ID Wing"],verbose=True)
print(f"  {GREEN if policy_ok else RED}{'OK' if policy_ok else 'FAIL'} {msg}{RESET}")


+-- PHASE 5: [C3] JIT Access + [B3] On-Chain Policy Update ------+

  [C3] JIT token grant -- ADM-SYS-001 needs time-limited elevated access:
  OK JIT token: JIT-329258  expires in 300s
    Grant event on-chain -- no standing admin privilege exists [C3].

    PRE-PREPARE [NODE-A->NODE-B]  Block #25  00dd0b7cf49741...
    PREPARE     [NODE-B->NODE-A]  OK ACK
    COMMIT      [BOTH NODES OK]  Block #25 finalised

  Block #025 [JIT-ADMIN]  00dd0b7cf49741e060...  [SIGNED]
  --------------------------------------------------------------------
  User    : ADM-SYS-001             Agency  : Bangladesh Computer Council
  Resource: military_personnel_db   Class   : TOP_SECRET
  IP      : 172.16.1.1              Geo     : Gov VPN
  Trust   : 0.960                   Decision: ALLOW
  ZT Gate : PASS        Risk    : |||||--------------- 2.7/10
  JIT     : JIT-329258
  Persona : [PERSONA_3_PRIVILEGED_ADMIN]
  Flags   : HIGH_SENSITIVITY(military_personnel_db), PRIVILEGED_ADMIN
  Volume  :         55 

### Phase 6: Trust decay over a session [C4 + B5]
The table shows effective trust as a 0.92 session ages. The same user is then re-evaluated 90 minutes into the session.

In [20]:
section("PHASE 6: [C4+B5] Session Continuity -- Trust Decay Over Time")
print(f"  {DIM}Interview 1 Q2: 'A user who passes ZTA at 09:00 and connects to a{RESET}")
print(f"  {DIM}public network at 10:30 receives no re-evaluation in your design.'{RESET}\n")

base_tr = 0.92
print(f"  {'Elapsed (min)':<18} {'Effective Trust':<18} {'Status'}")
print(f"  {'-'*16}  {'-'*16}  {'-'*26}")
for elapsed in [0, 15, 30, 60, 90, 120]:
    eff = decay_trust(base_tr, elapsed)
    ok  = eff >= ZeroTrustGateway.DEVICE_TRUST_THRESHOLD
    status = (GREEN + "COMPLIANT" if ok else RED + "RE-AUTH REQUIRED") + RESET
    print(f"  {elapsed:<18} {eff:<18.3f} {status}")

print(f"\n  {DIM}USR-MoF-001 at 90-min session -- decayed trust re-evaluated:{RESET}")
decay_blk,zt = engine.process("USR-MoF-001","Ministry of Finance","tax_records",
                         "10.0.1.45",0.92,10,50.0,elapsed_min=90.0,
                         persona="TRUST_DECAY_DEMO")
print_block(decay_blk,zt,"DECAYED")


+-- PHASE 6: [C4+B5] Session Continuity -- Trust Decay Over Time +
  Interview 1 Q2: 'A user who passes ZTA at 09:00 and connects to a
  public network at 10:30 receives no re-evaluation in your design.'

  Elapsed (min)      Effective Trust    Status
  ----------------  ----------------  --------------------------
  0                  0.920              COMPLIANT
  15                 0.864              COMPLIANT
  30                 0.819              COMPLIANT
  60                 0.747              COMPLIANT
  90                 0.691              RE-AUTH REQUIRED
  120                0.645              RE-AUTH REQUIRED

  USR-MoF-001 at 90-min session -- decayed trust re-evaluated:


  Block #027 [DECAYED]  00ff43fd23f2cfa139...  [SIGNED]
  --------------------------------------------------------------------
  User    : USR-MoF-001             Agency  : Ministry of Finance
  Resource: tax_records             Class   : CONFIDENTIAL
  IP      : 10.0.1.45               Geo     : Bangladesh Internal
  Trust   : 0.691                   Decision: ZTA_CHALLENGE
  ZT Gate : CHALLENGE   Risk    : |||----------------- 1.6/10
  Persona : [TRUST_DECAY_DEMO]
  Flags   : ZTA_DEVICE_CHALLENGE
  Volume  :         50 KB  Event: AUTH_REQUEST


### Phase 7: Attacker tampers with the log on Node-A
The attacker edits the four Persona 1 records on Node-A to make the blocked requests look like normal, allowed ones. Node-B's copies are untouched.

In [21]:
section("PHASE 7: Attacker Covers Tracks -- Tamper Attack Blocks on Node-A")
print(f"  {DIM}Modifies: decision, zt_verdict, IP, risk_score, data_volume.{RESET}\n")

pre: Dict[int, dict] = {}
for blk in attack_blocks:
    pre[blk.index] = {k: getattr(blk.log_entry,k) for k in
                      ("decision","zt_verdict","ip_address","risk_score","data_volume_kb")}
    blk.tamper("decision","ALLOW"); blk.tamper("zt_verdict","PASS")
    blk.tamper("ip_address","10.0.1.99"); blk.tamper("risk_score",1.1)
    blk.tamper("data_volume_kb",12.0); blk.tamper("risk_level","LOW")
    blk.tamper("risk_factors","NONE")
    p = pre[blk.index]
    print(f"  {RED}TAMPERED Block #{blk.index:03d}:{RESET}  "
          f"decision {RED}{p['decision']}{RESET}->{YELLOW}ALLOW{RESET}  "
          f"zt_verdict {RED}{p['zt_verdict']}{RESET}->{YELLOW}PASS{RESET}  "
          f"vol {RED}{p['data_volume_kb']:.0f}KB{RESET}->{YELLOW}12KB{RESET}")


+-- PHASE 7: Attacker Covers Tracks -- Tamper Attack Blocks on Node-A +
  Modifies: decision, zt_verdict, IP, risk_score, data_volume.

  TAMPERED Block #009:  decision ZTA_BLOCK->ALLOW  zt_verdict BLOCK->PASS  vol 2KB->12KB
  TAMPERED Block #010:  decision ZTA_BLOCK->ALLOW  zt_verdict BLOCK->PASS  vol 20KB->12KB
  TAMPERED Block #011:  decision ZTA_BLOCK->ALLOW  zt_verdict BLOCK->PASS  vol 15000KB->12KB
  TAMPERED Block #012:  decision ZTA_BLOCK->ALLOW  zt_verdict BLOCK->PASS  vol 22000KB->12KB


### Phase 8: Chain validation, cross-node comparison, heartbeats and signatures
Both nodes are validated independently, then compared block by block. The heartbeat check confirms no completeness anchors are missing, and a sample of signatures is verified.

In [22]:
section("PHASE 8: [B2] Chain Validation + [C7] Heartbeat Gaps + [B1] ECDSA Check")

print(f"\n  {BOLD}[B2] Node-A (tampered primary):{RESET}")
va,tampered_a = node_a.validate_full_chain()
print(f"  {'FAIL' if not va else 'OK'} Node-A: "
      f"{RED if not va else GREEN}{'VIOLATED -- blocks: '+str(tampered_a) if not va else 'VALID'}{RESET}")
if not va:
    for idx in tampered_a:
        blk  = node_a.chain[idx]; orig = pre.get(idx,{}); recomp = blk._compute_hash()
        print(f"\n    {RED}Block #{idx:03d} -- FORENSIC EVIDENCE:{RESET}")
        print(f"      Stored hash    : {CYAN}{blk.hash[:38]}...{RESET}")
        print(f"      Recomputed hash: {RED}{recomp[:38]}...{RESET}  <- MISMATCH")
        if orig:
            print(f"      ORIGINAL decision   : {RED}{orig.get('decision','?')}{RESET} (falsified -> ALLOW)")
            print(f"      ORIGINAL zt_verdict : {RED}{orig.get('zt_verdict','?')}{RESET} (ZTA block erased)")
            print(f"      ORIGINAL volume     : {RED}{orig.get('data_volume_kb',0):.0f}KB{RESET} (falsified -> 12KB)")

print(f"\n  {BOLD}[B2] Node-B (replica):{RESET}")
vb,tampered_b = node_b.validate_full_chain()
print(f"  {'OK' if vb else 'FAIL'} Node-B: "
      f"{GREEN if vb else RED}{'VALID' if vb else 'VIOLATED -- blocks: '+str(tampered_b)}{RESET}")

divergent = node_a.diverges_from(node_b)
print(f"\n  {BOLD}[B2] Cross-node comparison (Node-A recomputed vs Node-B stored):{RESET}")
if divergent:
    print(f"  {RED}Nodes disagree at blocks {divergent} -- Node-A's copies were altered{RESET}")
    print(f"  {DIM}  Node-B's original records are the evidence; the attacker would need both nodes.{RESET}")
else:
    print(f"  {GREEN}Nodes agree on every block{RESET}")

print(f"\n  {BOLD}[C7] Heartbeat gap detection on Node-B:{RESET}")
gaps = node_b.detect_heartbeat_gaps(interval=PBFTConsensus.HEARTBEAT_EVERY)
hb_ct = sum(1 for b in node_b.chain if b.log_entry.event_type=="SYSTEM_HEARTBEAT")
if gaps: print(f"  {RED}WARNING Missing heartbeat near positions: {gaps} -- possible log omission{RESET}")
else:    print(f"  {GREEN}OK All heartbeat anchors present ({hb_ct} blocks) -- no gap detected{RESET}")

print(f"\n  {BOLD}[B1] ECDSA spot-check on Node-B blocks 2-5:{RESET}")
for blk in node_b.chain[2:6]:
    eb = json.dumps(asdict(blk.log_entry), sort_keys=True).encode()
    ok = keys.verify(blk.signer_id, eb, blk.signature)
    sc = GREEN if ok else RED
    print(f"    Block #{blk.index:03d}  signer:{blk.signer_id:<22}  ECDSA: {sc}{'VALID' if ok else 'INVALID'}{RESET}")


+-- PHASE 8: [B2] Chain Validation + [C7] Heartbeat Gaps + [B1] ECDSA Check +

  [B2] Node-A (tampered primary):
  FAIL Node-A: VIOLATED -- blocks: [9, 10, 11, 12]

    Block #009 -- FORENSIC EVIDENCE:
      Stored hash    : 0001e7cfc5aef9bfe120eb2c35d3f0360a1b61...
      Recomputed hash: 50431e444261989c2b0924515d68ffd9235b42...  <- MISMATCH
      ORIGINAL decision   : ZTA_BLOCK (falsified -> ALLOW)
      ORIGINAL zt_verdict : BLOCK (ZTA block erased)
      ORIGINAL volume     : 2KB (falsified -> 12KB)

    Block #010 -- FORENSIC EVIDENCE:
      Stored hash    : 0049cafa1f1bb074dfda1a4aa1eb2913a866a5...
      Recomputed hash: 5548a48c3686cdf6294ee4a5ff5e418a0ef2f4...  <- MISMATCH
      ORIGINAL decision   : ZTA_BLOCK (falsified -> ALLOW)
      ORIGINAL zt_verdict : BLOCK (ZTA block erased)
      ORIGINAL volume     : 20KB (falsified -> 12KB)

    Block #011 -- FORENSIC EVIDENCE:
      Stored hash    : 001c3d6b3974604303a190795cfd23415a53c6...
      Recomputed hash: d753d10a89e123d045

### Phase 9: Insider tries to fabricate an ALLOW record [B1]
Someone with node access forges a record claiming the attacker was allowed, signing it with the wrong key. The replica rejects it during the PREPARE step.

In [23]:
section("PHASE 9: [B1] ECDSA -- Admin Attempts to Fabricate an ALLOW Block")
print(f"  {DIM}Interview 2 Q3: 'A sophisticated insider who controls the logging process{RESET}")
print(f"  {DIM}would write the false record before the hash is computed -- your prototype{RESET}")
print(f"  {DIM}would not catch that.' [B1] closes this: ECDSA signature required.\n{RESET}")

fake_entry = ZTALogEntry(
    event_type="AUTH_REQUEST", user_id="ATCK-EXT-001",
    agency="National ID Wing", resource="citizen_nid_db",
    ip_address="10.0.0.1", geo_location="Bangladesh Internal",
    device_trust_score=0.95, data_classification="TOP_SECRET",
    risk_score=0.8, risk_level="MINIMAL", risk_factors="NONE",
    decision="ALLOW", zt_verdict="PASS", zt_deny_reason="",
    timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
    session_id="FABRICATED-0001", threat_persona="INSIDER_FABRICATION")

fake_bytes = json.dumps(asdict(fake_entry), sort_keys=True).encode()
wrong_sig  = keys.sign("ADM-SYS-001", fake_bytes)  # wrong key
fake_block = Block(len(node_b.chain), fake_entry, node_b.head_hash(),
                   wrong_sig, "ATCK-EXT-001")
fake_block.mine(BlockchainNode.DIFFICULTY)

fab_ok, fab_reason = node_b.validate_candidate(fake_block, keys)
print(f"  Fabricated block -- claimed signer: ATCK-EXT-001 (not registered)")
print(f"  Signature used   : ADM-SYS-001 private key (wrong key, wrong identity)")
if fab_ok:
    print(f"  PREPARE          : {RED}ACCEPTED -- fabrication NOT detected{RESET}")
else:
    print(f"  PREPARE          : {GREEN}REJECTED -- {fab_reason}{RESET}")
    print(f"  {GREEN}OK Without the claimed user's private key, fabrication is blocked [B1].{RESET}")


+-- PHASE 9: [B1] ECDSA -- Admin Attempts to Fabricate an ALLOW Block +
  Interview 2 Q3: 'A sophisticated insider who controls the logging process
  would write the false record before the hash is computed -- your prototype
  would not catch that.' [B1] closes this: ECDSA signature required.

  Fabricated block -- claimed signer: ATCK-EXT-001 (not registered)
  Signature used   : ADM-SYS-001 private key (wrong key, wrong identity)
  PREPARE          : REJECTED -- ECDSA_SIGNATURE_INVALID
  OK Without the claimed user's private key, fabrication is blocked [B1].


### Phase 10: Risk scores for all access decisions
Read from Node-B, the untampered replica.

In [24]:
section("PHASE 10: Risk Score Analysis -- All Access Decisions (Node-B)")
all_blks = [b for b in node_b.chain[1:]
            if b.log_entry.event_type not in
            ("SYSTEM_HEARTBEAT","POLICY_INIT","POLICY_UPDATE","JIT_GRANT")]
decisions = [b.log_entry.decision   for b in all_blks]
levels    = [b.log_entry.risk_level for b in all_blks]
scores    = [b.log_entry.risk_score for b in all_blks]

print(f"\n  {'Block':<8} {'User':<20} {'Risk Score':<24} {'Decision':<16} {'ZT Gate'}")
print(f"  {'-'*7}  {'-'*19}  {'-'*22}  {'-'*14}  {'-'*10}")
for blk in all_blks:
    e   = blk.log_entry
    dc  = GREEN if e.decision=="ALLOW" else (YELLOW if "CHALLENGE" in e.decision else RED)
    ztc = GREEN if e.zt_verdict=="PASS" else (YELLOW if e.zt_verdict=="CHALLENGE" else RED)
    print(f"  #{blk.index:<6} {e.user_id:<20} {bar(e.risk_score,14):<24} "
          f"{dc}{e.decision:<16}{RESET} {ztc}{e.zt_verdict}{RESET}")


+-- PHASE 10: Risk Score Analysis -- All Access Decisions (Node-B) +

  Block    User                 Risk Score               Decision         ZT Gate
  -------  -------------------  ----------------------  --------------  ----------
  #2      USR-MoF-001          |------------- 1.3/10    ALLOW            PASS
  #3      USR-NID-007          |||----------- 2.5/10    ALLOW            PASS
  #4      USR-BB-003           ||------------ 1.8/10    ALLOW            PASS
  #5      USR-EC-002           ||------------ 1.9/10    ALLOW            PASS
  #6      USR-MoH-009          |------------- 1.3/10    ALLOW            PASS
  #7      USR-BCC-011          |------------- 0.8/10    ALLOW            PASS
  #9      ATCK-EXT-001         ||------------ 1.6/10    ZTA_BLOCK        BLOCK
  #10     ATCK-EXT-001         |||||--------- 3.8/10    ZTA_BLOCK        BLOCK
  #11     ATCK-EXT-001         ||||||||||||-- 8.6/10    ZTA_BLOCK        BLOCK
  #12     ATCK-EXT-001         ||||||||||||-- 8.9/10    ZTA

### Phase 11: Summary
Every figure below is computed from Node-B's records and the checks above.

In [25]:
section("PHASE 11: B-ZTA v4.1 -- Combined Defence Summary")

def persona_decisions(p):
    return [b.log_entry.decision for b in all_blks if b.log_entry.threat_persona == p]

legit_dec = [b.log_entry.decision for b in all_blks if not b.log_entry.threat_persona]
ext_dec   = persona_decisions("PERSONA_1_EXTERNAL")
ins_dec   = persona_decisions("PERSONA_2_INSIDER")
stl_dec   = persona_decisions("PERSONA_4_STOLEN_CRED")
su_denied = sum(1 for b in all_blks if "STEP_UP_FAILED" in b.log_entry.risk_factors)
admin_dec = node_b.chain[admin_blk.index].log_entry.decision
decay_dec = node_b.chain[decay_blk.index].log_entry.decision

signed_ct = sum(1 for b in node_b.chain if keys.verify(
    b.signer_id, json.dumps(asdict(b.log_entry), sort_keys=True).encode(), b.signature))
critical  = levels.count("CRITICAL")
avg_score = sum(scores)/len(scores) if scores else 0
total_kb  = sum(b.log_entry.data_volume_kb for b in node_b.chain)
detect_pct = 100*len(set(tampered_a) & {b.index for b in attack_blocks})/len(attack_blocks)

print(f"""
  {BOLD}+== LAYER 1: Zero Trust Gateway + Risk Engine (access requests) =={RESET}
    Legitimate requests allowed (Phase 1)                     : {legit_dec.count('ALLOW')}/{len(legit_dec)}
    External attacker requests stopped (Persona 1)            : {sum(d!='ALLOW' for d in ext_dec)}/{len(ext_dec)}
    Insider requests denied after failed step-up (Persona 2)  : {ins_dec.count('DENY')}/{len(ins_dec)}
    Stolen-credential requests blocked at MFA (Persona 4)     : {stl_dec.count('ZTA_BLOCK')}/{len(stl_dec)}
    Admin request with valid JIT token (Persona 3)            : {admin_dec}
    90-minute session after trust decay                       : {decay_dec}
    Decisions -- ALLOW {decisions.count('ALLOW')} | DENY {decisions.count('DENY')} | ZTA_BLOCK {decisions.count('ZTA_BLOCK')} | ZTA_CHALLENGE {decisions.count('ZTA_CHALLENGE')}
    CRITICAL risk events                                      : {critical}
    Average risk score                                        : {avg_score:.2f}/10

  {BOLD}+== LAYER 2: Blockchain Audit -- Dual-Node Consensus [B2] =={RESET}
    Total blocks Node-A / Node-B                              : {len(node_a.chain)} / {len(node_b.chain)}
    Blocks with a valid ECDSA signature [B1]                  : {signed_ct}/{len(node_b.chain)} (genesis is unsigned)
    Heartbeat anchors / gaps detected [C7]                    : {hb_ct} / {len(gaps)}
    Policy blocks on-chain [B3]                               : {sum(1 for b in node_b.chain if 'POLICY' in b.log_entry.event_type)}
    JIT grant records [C3]                                    : {sum(1 for b in node_b.chain if b.log_entry.event_type=='JIT_GRANT')}
    Tampered blocks detected on Node-A                        : {len(tampered_a)}/{len(attack_blocks)} ({detect_pct:.0f}%)
    Node-B still valid after the attack                       : {vb}
    Fabricated block rejected [B1]                            : {not fab_ok} ({fab_reason})
    Total data volume logged                                  : {total_kb/1024:.1f} MB

  {BOLD}Expert interview enhancements:{RESET}
    [B1] ECDSA signing     -- entries signed by the requester; forgery rejected
    [B2] Dual-node copies  -- tampering on Node-A exposed by Node-B comparison
    [B3] On-chain policy   -- changes endorsed by both nodes, applied after commit
    [B4] Post-quantum      -- ML-DSA / Dilithium documented as forward path (not implemented)
    [C1] Threat personas   -- external / insider / privileged admin / stolen credentials
    [C2] Step-up enforced  -- insider not enrolled -> DENY, not silent pass-through
    [C3] JIT access        -- admin requests require a valid, time-limited token
    [C4] Trust decay       -- 0.92 session drops below 0.70 at ~85 min -> re-challenged
    [C5] Semantic velocity -- 6-resource sweep forces step-up
    [C6] Reputation EMA    -- recent behaviour weighted in the risk score
    [C7] Heartbeat anchors -- missing anchors reveal suppressed logging
    [C8] Control plane     -- gateway logically separated from the audit logger
""")
sep()


+-- PHASE 11: B-ZTA v4.1 -- Combined Defence Summary ------------+

  +== LAYER 1: Zero Trust Gateway + Risk Engine (access requests) ==
    Legitimate requests allowed (Phase 1)                     : 6/6
    External attacker requests stopped (Persona 1)            : 4/4
    Insider requests denied after failed step-up (Persona 2)  : 2/7
    Stolen-credential requests blocked at MFA (Persona 4)     : 2/2
    Admin request with valid JIT token (Persona 3)            : ALLOW
    90-minute session after trust decay                       : ZTA_CHALLENGE
    Decisions -- ALLOW 12 | DENY 2 | ZTA_BLOCK 6 | ZTA_CHALLENGE 1
    CRITICAL risk events                                      : 2
    Average risk score                                        : 2.95/10

  +== LAYER 2: Blockchain Audit -- Dual-Node Consensus [B2] ==
    Total blocks Node-A / Node-B                              : 28 / 28
    Blocks with a valid ECDSA signature [B1]                  : 27/28 (genesis is unsigned)
    Heart

## 13. Self-checks

Assertions that verify the simulation's main claims. If a code change breaks one of them, this cell fails and says which claim no longer holds.

In [26]:
checks = []
def check(name, condition):
    checks.append((name, bool(condition)))
    print(f"  {'PASS' if condition else 'FAIL'}  {name}")

print("Self-checks")
check("All tampered Node-A blocks detected",
      set(tampered_a) == {b.index for b in attack_blocks})
check("Node-B remains valid after tampering on Node-A", vb)
check("Cross-node comparison pinpoints exactly the tampered blocks",
      divergent == sorted(b.index for b in attack_blocks))
check("Fabricated block rejected for an invalid signature",
      not fab_ok and fab_reason == "ECDSA_SIGNATURE_INVALID")
check("All legitimate Phase 1 requests allowed",
      legit_dec.count("ALLOW") == len(legit_dec))
check("No external attacker request allowed", "ALLOW" not in ext_dec)
check("Insider's last two requests denied after failed step-up",
      ins_dec[-2:] == ["DENY", "DENY"])
check("Stolen-credential requests blocked at MFA",
      stl_dec == ["ZTA_BLOCK", "ZTA_BLOCK"])
check("Admin with a valid JIT token allowed", admin_dec == "ALLOW")
check("JIT token rejected for a different user",
      jit_mgr.validate(tid, "USR-MoF-001", "military_personnel_db")[0] is False)
check("Trust stays compliant at 60 min and drops below threshold at 90 min",
      decay_trust(0.92, 60) >= ZeroTrustGateway.DEVICE_TRUST_THRESHOLD > decay_trust(0.92, 90))
check("90-minute session re-challenged by the gateway", decay_dec == "ZTA_CHALLENGE")
check("No false heartbeat alarm on the real chain", gaps == [])

# Negative test: a fresh chain with the heartbeat monitor disabled must be flagged
t_keys = ECDSAKeyRegistry(); t_keys.register("SYSTEM")
t_a = BlockchainNode("T-A", "test", True); t_b = BlockchainNode("T-B", "test")
t_b.chain = copy.deepcopy(t_a.chain)
t_pbft = PBFTConsensus(t_a, t_b, t_keys); t_pbft.HEARTBEAT_EVERY = 10**9  # monitor off
for i in range(10):
    e = ZTALogEntry(event_type="AUTH_REQUEST", user_id="SYSTEM", agency="test",
                    resource="staff_directory", ip_address="10.0.0.1",
                    geo_location="Bangladesh Internal", device_trust_score=0.9,
                    data_classification="INTERNAL", risk_score=0.5, risk_level="MINIMAL",
                    risk_factors="NONE", decision="ALLOW", zt_verdict="PASS",
                    zt_deny_reason="", timestamp="test", session_id=f"T-{i}")
    t_pbft.add_block(e, t_keys.sign("SYSTEM", json.dumps(asdict(e), sort_keys=True).encode()), "SYSTEM")
check("Heartbeat detector flags a chain whose heartbeat monitor was disabled",
      t_b.detect_heartbeat_gaps(interval=7) != [])

passed = sum(ok for _, ok in checks)
print(f"\n{passed}/{len(checks)} checks passed")
assert passed == len(checks), "One or more claims no longer hold -- see FAIL lines above"

Self-checks
  PASS  All tampered Node-A blocks detected
  PASS  Node-B remains valid after tampering on Node-A
  PASS  Cross-node comparison pinpoints exactly the tampered blocks
  PASS  Fabricated block rejected for an invalid signature
  PASS  All legitimate Phase 1 requests allowed
  PASS  No external attacker request allowed
  PASS  Insider's last two requests denied after failed step-up
  PASS  Stolen-credential requests blocked at MFA
  PASS  Admin with a valid JIT token allowed
  PASS  JIT token rejected for a different user
  PASS  Trust stays compliant at 60 min and drops below threshold at 90 min
  PASS  90-minute session re-challenged by the gateway
  PASS  No false heartbeat alarm on the real chain


  PASS  Heartbeat detector flags a chain whose heartbeat monitor was disabled

14/14 checks passed
